In [3]:
import platform, os
print(platform.system())
print(round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1), "GB RAM")

Linux
13.6 GB RAM


In [4]:
! pip install -q --upgrade kaggle duckdb implicit

In [5]:
import os, getpass
from pathlib import Path

token = getpass.getpass("Paste your Kaggle token: ").strip()
print("token length:", len(token))

os.environ["KAGGLE_API_TOKEN"] = token
kdir = Path.home() / ".kaggle"
kdir.mkdir(exist_ok=True)
(kdir / "access_token").write_text(token)
os.chmod(kdir / "access_token", 0o600)

token length: 37


In [6]:
COMP = "h-and-m-personalized-fashion-recommendations"
for f in ["transactions_train.csv", "customers.csv", "articles.csv"]:
    !kaggle competitions download -c {COMP} -f {f} -p /content/raw -q
! cd /content/raw && unzip -o -q '*.zip' && rm -f *.zip
! ls -lh /content/raw


3 archives were successfully processed.
total 3.5G
-rw-r--r-- 1 root root  35M Jan 17  2022 articles.csv
-rw-r--r-- 1 root root 198M Jan 17  2022 customers.csv
-rw-r--r-- 1 root root 3.3G Jan 17  2022 transactions_train.csv


In [7]:
!rm -rf /content/repo
!git clone -q https://github.com/WoahImCoding/ecommerce-recommender.git /content/repo
!ls /content/repo/src

import sys
sys.path.append("/content/repo")
from src.metrics import evaluate
print("metrics imported")

metrics.py
metrics imported


Main Table creation

In [8]:
import duckdb
from pathlib import Path

raw, proc = Path("/content/raw"), Path("/content/processed")
proc.mkdir(exist_ok=True)

con = duckdb.connect()
for name in ["transactions_train", "customers", "articles"]:
    con.execute(f"COPY (SELECT * FROM read_csv_auto('{raw}/{name}.csv')) TO '{proc}/{name}.parquet' (FORMAT PARQUET)")

con.execute(f"CREATE VIEW transactions AS SELECT * FROM '{proc}/transactions_train.parquet'")
con.execute(f"CREATE VIEW articles AS SELECT * FROM '{proc}/articles.parquet'")
con.execute("""
    CREATE OR REPLACE VIEW tx AS
    SELECT *, CAST(date_diff('day', t_dat, (SELECT max(t_dat) FROM transactions)) // 7 AS INTEGER) AS week_ago
    FROM transactions
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
con.sql("SELECT count(*) AS n FROM tx").df()

,n
0,31788324


In [10]:
con.execute("""
    CREATE OR REPLACE VIEW tx AS
    SELECT *,
           CAST(date_diff('day', t_dat, (SELECT max(t_dat) FROM transactions)) // 7 AS INTEGER) AS week_ago
    FROM transactions
""")

con.sql("""
    SELECT week_ago,
           min(t_dat)                  AS first_day,
           max(t_dat)                  AS last_day,
           count(*)                    AS n_tx,
           count(DISTINCT customer_id) AS n_customers
    FROM tx
    WHERE week_ago <= 4
    GROUP BY 1
    ORDER BY 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,week_ago,first_day,last_day,n_tx,n_customers
0,0,2020-09-16,2020-09-22,240311,68984
1,1,2020-09-09,2020-09-15,255241,72019
2,2,2020-09-02,2020-09-08,264657,75822
3,3,2020-08-26,2020-09-01,283181,80253
4,4,2020-08-19,2020-08-25,256644,72035


In [11]:
# Give every customer a small integer ID
con.execute("""
    CREATE OR REPLACE TABLE cust_map AS
    SELECT customer_id,
           CAST(row_number() OVER (ORDER BY customer_id) - 1 AS INTEGER) AS cust
    FROM (SELECT DISTINCT customer_id FROM tx)
""")

# Load only the last 16 weeks, with compact column types
recent = con.sql("""
    SELECT m.cust,
           CAST(t.article_id AS INTEGER)       AS article_id,
           t.t_dat,
           CAST(t.week_ago AS SMALLINT)        AS week_ago,
           CAST(t.price AS FLOAT)              AS price,
           CAST(t.sales_channel_id AS TINYINT) AS channel
    FROM tx t
    JOIN cust_map m USING (customer_id)
    WHERE t.week_ago <= 15
""").df()

print(f"{len(recent):,} rows, {recent.memory_usage(deep=True).sum() / 1e6:.0f} MB")
recent.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

5,063,720 rows, 116 MB


,cust,article_id,t_dat,week_ago,price,channel
0,427153,750330002,2020-06-04,15,0.050831,2
1,427153,750330002,2020-06-04,15,0.050831,2
2,427153,888206001,2020-06-04,15,0.033881,2
3,427153,752358002,2020-06-04,15,0.025407,2
4,427153,882354001,2020-06-04,15,0.033881,2


In [12]:
labels = con.sql("""
    SELECT customer_id, list(DISTINCT article_id) AS items
    FROM tx WHERE week_ago = 0 GROUP BY 1
""").df()
actual = dict(zip(labels["customer_id"], labels["items"].apply(set)))

recent_top = con.sql("""
    SELECT article_id FROM tx WHERE week_ago = 1
    GROUP BY 1 ORDER BY count(*) DESC LIMIT 12
""").df()["article_id"].tolist()

print(len(actual), evaluate(lambda c: recent_top, actual))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

68984 {'recall@12': np.float64(0.0255), 'map@12': np.float64(0.0087)}


Test - Train Split

In [13]:
import numpy as np
import pandas as pd

test = recent[recent["week_ago"] == 0]                  # the week we predict
hist = recent[recent["week_ago"].between(1, 12)]        # the 12 weeks before it

actual = test.groupby("cust")["article_id"].apply(set).to_dict()
print(len(actual), "test customers")                    # should be 68984

68984 test customers


In [14]:
rep = (hist.groupby(["cust", "article_id"])
           .agg(last_day=("t_dat", "max"), n_bought=("t_dat", "size"))
           .reset_index()
           .sort_values(["cust", "last_day", "n_bought"], ascending=[True, False, False]))
rep["rank"] = rep.groupby("cust").cumcount()
rep = rep[rep["rank"] < 30]

rep_lists = rep.groupby("cust")["article_id"].apply(list).to_dict()
print(len(rep_lists), "customers have repurchase candidates")

505310 customers have repurchase candidates


In [15]:
trend = hist.loc[hist["week_ago"] == 1, "article_id"].value_counts().index[:30].tolist()

Measure candidate recall

In [16]:
def candidate_recall(get_cands, actual):
    """Share of each customer's purchases that appear anywhere in their candidate list."""
    vals = [len(items & set(get_cands(c))) / len(items) for c, items in actual.items()]
    return round(float(np.mean(vals)), 4)

def union_cands(c):
    return set(rep_lists.get(c, [])) | set(trend)

pd.DataFrame({
    "candidate_recall": [
        candidate_recall(lambda c: rep_lists.get(c, []), actual),
        candidate_recall(lambda c: trend, actual),
        candidate_recall(union_cands, actual),
    ],
    "avg_candidates": [
        np.mean([len(rep_lists.get(c, [])) for c in actual]),
        len(trend),
        np.mean([len(union_cands(c)) for c in actual]),
    ],
}, index=["Repurchase (up to 30)", "Trending (30)", "Both combined"]).round(4)

,candidate_recall,avg_candidates
Repurchase (up to 30),0.0378,6.6337
Trending (30),0.0511,30.0000
Both combined,0.0862,36.5205


Build the Customer x item matrix

In [17]:
import scipy.sparse as sp

hist = hist.copy()
hist["t_dat"] = pd.to_datetime(hist["t_dat"])

users = hist["cust"].unique()
items = hist["article_id"].unique()
u_idx = pd.Series(np.arange(len(users)), index=users)   # customer ID -> matrix row
i_idx = pd.Series(np.arange(len(items)), index=items)   # article ID  -> matrix column

# Recent purchases count more: weight halves every 28 days
age_days = (hist["t_dat"].max() - hist["t_dat"]).dt.days.to_numpy()
weight = (0.5 ** (age_days / 28)).astype(np.float32)

mat = sp.csr_matrix(
    (weight, (u_idx[hist["cust"]].to_numpy(), i_idx[hist["article_id"]].to_numpy())),
    shape=(len(users), len(items)),
)
print(mat.shape, f"{mat.nnz:,} non-zero cells")

(505310, 41376) 3,237,697 non-zero cells


Train ALS: Alternating Least Squares

In [18]:
from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.05, random_state=42)
als.fit(mat)

/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

Top 30 ALS Candidates for each test customer

In [19]:
test_users = [c for c in actual if c in u_idx.index]       # ALS only knows customers with history
rows = u_idx[test_users].to_numpy()
ids, scores = als.recommend(rows, mat[rows], N=30, filter_already_liked_items=False)

als_lists = {c: items[ids[k]].tolist() for k, c in enumerate(test_users)}
print(len(als_lists), "customers have ALS candidates")

49219 customers have ALS candidates


ALS to the candidate recall table

In [18]:
def union3(c):
    return set(rep_lists.get(c, [])) | set(trend) | set(als_lists.get(c, []))

pd.DataFrame({
    "candidate_recall": [
        candidate_recall(lambda c: rep_lists.get(c, []), actual),
        candidate_recall(lambda c: trend, actual),
        candidate_recall(lambda c: als_lists.get(c, []), actual),
        candidate_recall(union_cands, actual),
        candidate_recall(union3, actual),
    ],
    "avg_candidates": [
        np.mean([len(rep_lists.get(c, [])) for c in actual]),
        len(trend),
        np.mean([len(als_lists.get(c, [])) for c in actual]),
        np.mean([len(union_cands(c)) for c in actual]),
        np.mean([len(union3(c)) for c in actual]),
    ],
}, index=["Repurchase", "Trending", "ALS", "Repurchase + trending", "All three"]).round(4)

,candidate_recall,avg_candidates
Repurchase,0.0378,6.6337
Trending,0.0511,30.0000
ALS,0.0377,21.4045
Repurchase + trending,0.0862,36.5205
All three,0.1066,55.4266


Items bought together: recommend that to the customers already purchased one item from the bucket

In [20]:
baskets = hist.loc[hist["week_ago"] <= 4, ["cust", "t_dat", "article_id"]].drop_duplicates()
con.register("b", baskets)

cop = con.sql("""
    WITH pairs AS (
        SELECT x.article_id AS src, y.article_id AS dst, count(*) AS n
        FROM b x
        JOIN b y
          ON x.cust = y.cust AND x.t_dat = y.t_dat AND x.article_id <> y.article_id
        GROUP BY 1, 2
    ),
    src_total AS (
        SELECT article_id AS src, count(*) AS n_src FROM b GROUP BY 1
    )
    SELECT src, dst, CAST(n AS DOUBLE) / n_src AS p
    FROM pairs JOIN src_total USING (src)
    QUALIFY row_number() OVER (PARTITION BY src ORDER BY n DESC, dst) <= 10
""").df()

print(f"{len(cop):,} item pairs")

235,832 item pairs


Pairs into Candidates for Each customers

In [21]:
last5 = (hist.sort_values("t_dat", ascending=False)
             .drop_duplicates(["cust", "article_id"])
             .groupby("cust").head(5)[["cust", "article_id"]])
last5 = last5[last5["cust"].isin(list(actual))]

m = last5.merge(cop, left_on="article_id", right_on="src")
s = (m.groupby(["cust", "dst"])["p"].sum()
      .reset_index()
      .sort_values(["cust", "p"], ascending=[True, False]))
cop_lists = s.groupby("cust").head(20).groupby("cust")["dst"].apply(list).to_dict()
print(len(cop_lists), "customers have co-purchase candidates")

49122 customers have co-purchase candidates


In [22]:
week1 = hist.loc[hist["week_ago"] == 1, "article_id"].value_counts().index
trend60, trend100 = week1[:60].tolist(), week1[:100].tolist()

def union_with(extra_trend, use_cop):
    def f(c):
        s = set(rep_lists.get(c, [])) | set(extra_trend) | set(als_lists.get(c, []))
        if use_cop:
            s |= set(cop_lists.get(c, []))
        return s
    return f

variants = {
    "All three (trending 30)":           union_with(trend, False),
    "+ co-purchase (trending 30)":       union_with(trend, True),
    "+ co-purchase, trending 60":        union_with(trend60, True),
    "+ co-purchase, trending 100":       union_with(trend100, True),
}
pd.DataFrame({
    "candidate_recall": [candidate_recall(f, actual) for f in variants.values()],
    "avg_candidates":   [np.mean([len(f(c)) for c in actual]) for f in variants.values()],
}, index=variants.keys()).round(4)

,candidate_recall,avg_candidates
All three (trending 30),0.1064,55.4574
+ co-purchase (trending 30),0.1203,65.8259
"+ co-purchase, trending 60",0.1458,94.0153
"+ co-purchase, trending 100",0.1765,131.3915


Function That builds candidates for any week

In [24]:
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares

def build_candidates(recent, label_week, n_trend=100, lookback=12, n_rep=30, n_als=30, n_cop=20):
    """Candidates (+ labels) for one week. History = the `lookback` weeks before label_week."""
    hist = recent[recent["week_ago"].between(label_week + 1, label_week + lookback)].copy()
    hist["t_dat"] = pd.to_datetime(hist["t_dat"])
    labels = recent.loc[recent["week_ago"] == label_week, ["cust", "article_id"]].drop_duplicates()
    custs = labels["cust"].unique()
    h = hist[hist["cust"].isin(custs)]

    # 1. Repurchase
    rep = (h.groupby(["cust", "article_id"])
             .agg(last_day=("t_dat", "max"), rep_count=("t_dat", "size"))
             .reset_index()
             .sort_values(["cust", "last_day", "rep_count"], ascending=[True, False, False]))
    rep["rep_rank"] = rep.groupby("cust").cumcount()
    rep = rep.loc[rep["rep_rank"] < n_rep, ["cust", "article_id", "rep_rank", "rep_count"]]

    # 2. Trending (same list for everyone)
    top = hist.loc[hist["week_ago"] == label_week + 1, "article_id"].value_counts().index[:n_trend].to_numpy()
    trend = pd.DataFrame({"cust": np.repeat(custs, len(top)),
                          "article_id": np.tile(top, len(custs)),
                          "trend_rank": np.tile(np.arange(len(top)), len(custs))})

    # 3. ALS
    users, items = hist["cust"].unique(), hist["article_id"].unique()
    u_idx = pd.Series(np.arange(len(users)), index=users)
    i_idx = pd.Series(np.arange(len(items)), index=items)
    age = (hist["t_dat"].max() - hist["t_dat"]).dt.days.to_numpy()
    w = (0.5 ** (age / 28)).astype(np.float32)
    mat = sp.csr_matrix((w, (u_idx[hist["cust"]].to_numpy(), i_idx[hist["article_id"]].to_numpy())),
                        shape=(len(users), len(items)))
    als = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.05, random_state=42)
    als.fit(mat, show_progress=False)
    warm = custs[np.isin(custs, users)]
    rows = u_idx[warm].to_numpy()
    ids, scores = als.recommend(rows, mat[rows], N=n_als, filter_already_liked_items=False)
    als_df = pd.DataFrame({"cust": np.repeat(warm, ids.shape[1]),
                           "article_id": items[ids.ravel()],
                           "als_rank": np.tile(np.arange(ids.shape[1]), len(warm)),
                           "als_score": scores.ravel()})

    # 4. Co-purchase
    baskets = hist.loc[hist["week_ago"] <= label_week + 4, ["cust", "t_dat", "article_id"]].drop_duplicates()
    con.register("b", baskets)
    pairs = con.sql("""
        WITH pairs AS (
            SELECT x.article_id AS src, y.article_id AS dst, count(*) AS n
            FROM b x JOIN b y
              ON x.cust = y.cust AND x.t_dat = y.t_dat AND x.article_id <> y.article_id
            GROUP BY 1, 2
        ),
        src_total AS (SELECT article_id AS src, count(*) AS n_src FROM b GROUP BY 1)
        SELECT src, dst, CAST(n AS DOUBLE) / n_src AS p
        FROM pairs JOIN src_total USING (src)
        QUALIFY row_number() OVER (PARTITION BY src ORDER BY n DESC, dst) <= 10
    """).df()
    last5 = (h.sort_values("t_dat", ascending=False)
               .drop_duplicates(["cust", "article_id"])
               .groupby("cust").head(5)[["cust", "article_id"]])
    cop = (last5.merge(pairs, left_on="article_id", right_on="src")
                .groupby(["cust", "dst"])["p"].sum().rename("cop_score").reset_index()
                .rename(columns={"dst": "article_id"})
                .sort_values(["cust", "cop_score"], ascending=[True, False]))
    cop = cop.groupby("cust").head(n_cop)

    # Combine: one row per (customer, candidate item), with every source's columns
    cands = pd.concat([d[["cust", "article_id"]] for d in (rep, trend, als_df, cop)]).drop_duplicates()
    for d in (rep, trend, als_df, cop):
        cands = cands.merge(d, on=["cust", "article_id"], how="left")
    for name, col in (("rep", "rep_rank"), ("trend", "trend_rank"), ("als", "als_rank"), ("cop", "cop_score")):
        cands[f"src_{name}"] = cands[col].notna().astype("int8")

    # Label: 1 if the customer bought this item in the label week
    cands = cands.merge(labels.assign(label=1), on=["cust", "article_id"], how="left")
    cands["label"] = cands["label"].fillna(0).astype("int8")
    cands["week"] = np.int8(label_week)
    floats = cands.select_dtypes("float64").columns
    cands[floats] = cands[floats].astype("float32")          # halve memory
    return cands, hist

In [25]:
cands0, hist0 = build_candidates(recent, label_week=0)

pos = cands0.groupby("cust")["label"].sum()
n_actual = pd.Series({c: len(a) for c, a in actual.items()})
print("candidate recall:", round(float((pos.reindex(n_actual.index, fill_value=0) / n_actual).mean()), 4))
print(f"{len(cands0):,} rows, {cands0['cust'].nunique():,} customers, "
      f"{len(cands0) / cands0['cust'].nunique():.0f} per customer, "
      f"{cands0.memory_usage(deep=True).sum() / 1e6:.0f} MB")
cands0.head()

candidate recall: 0.1764
9,064,055 rows, 68,984 customers, 131 per customer, 344 MB


,cust,article_id,rep_rank,rep_count,trend_rank,als_rank,als_score,cop_score,src_rep,src_trend,src_als,src_cop,label,week
0,86,889036004,0.0,1.0,NaN,NaN,NaN,NaN,1,0,0,0,0,0
1,86,880017001,1.0,2.0,NaN,NaN,NaN,NaN,1,0,0,0,0,0
2,86,621381012,2.0,1.0,NaN,NaN,NaN,0.157803,1,0,0,1,0,0
3,86,640021012,3.0,1.0,NaN,NaN,NaN,0.100000,1,0,0,1,0,0
4,86,904026001,4.0,1.0,NaN,NaN,NaN,NaN,1,0,0,0,0,0


# Phase 5: The LightGBM Ranker:
The plan: a model that looks at each customer's 131 candidates and re-orders them, so the item they'll actually buy move into the top 12. To learn that, it needs examples where we know the answers: week 1,2,3.

Customer and Articles Tables

In [26]:
cust_feat = con.sql("""
    SELECT m.cust,
           CAST(c.age AS DOUBLE)                 AS age,
           COALESCE(CAST(c.FN AS DOUBLE), 0)     AS fn,
           COALESCE(CAST(c.Active AS DOUBLE), 0) AS active,
           CASE c.club_member_status WHEN 'ACTIVE' THEN 0 WHEN 'PRE-CREATE' THEN 1
                                     WHEN 'LEFT CLUB' THEN 2 ELSE 3 END AS club,
           CASE c.fashion_news_frequency WHEN 'Regularly' THEN 2 WHEN 'Monthly' THEN 1 ELSE 0 END AS news
    FROM '/content/processed/customers.parquet' c
    JOIN cust_map m USING (customer_id)
""").df().set_index("cust").astype("float32")

art = con.sql("""
    SELECT CAST(article_id AS INTEGER)        AS article_id,
           CAST(product_type_no AS INTEGER)   AS product_type_no,
           CAST(colour_group_code AS INTEGER) AS colour_group_code,
           CAST(department_no AS INTEGER)     AS department_no,
           product_group_name, index_code
    FROM articles
""").df()
art["product_group_code"] = art["product_group_name"].astype("category").cat.codes
art["index_code_num"] = art["index_code"].astype("category").cat.codes
art = art.set_index("article_id")[["product_type_no", "colour_group_code", "department_no",
                                   "product_group_code", "index_code_num"]]
print(len(cust_feat), "customers,", len(art), "articles")

1362281 customers, 105542 articles


The Feature Function

In [27]:
def add_features(cands, hist, cust_feat, art):
    """Add user, item and user-item features, computed only from `hist` (the weeks before the label week)."""
    ref = hist["t_dat"].max() + pd.Timedelta(days=1)
    days_ago = (ref - hist["t_dat"]).dt.days

    # User features: how this customer shops
    g = hist.groupby("cust")
    user = pd.DataFrame({
        "u_n_tx": g.size(),
        "u_n_days": g["t_dat"].nunique(),
        "u_days_since_last": days_ago.groupby(hist["cust"]).min(),
        "u_avg_price": g["price"].mean(),
        "u_online_share": (hist["channel"] == 2).groupby(hist["cust"]).mean(),
    })

    # Item features: how this item is selling
    gi = hist.groupby("article_id")
    item = pd.DataFrame({
        "i_cnt_1w": hist[days_ago <= 7].groupby("article_id").size(),
        "i_cnt_4w": hist[days_ago <= 28].groupby("article_id").size(),
        "i_cnt_12w": gi.size(),
        "i_n_buyers": gi["cust"].nunique(),
        "i_avg_price": gi["price"].mean(),
        "i_days_since_first": days_ago.groupby(hist["article_id"]).max(),
        "i_mean_buyer_age": hist.assign(age=hist["cust"].map(cust_feat["age"])).groupby("article_id")["age"].mean(),
    })
    item[["i_cnt_1w", "i_cnt_4w"]] = item[["i_cnt_1w", "i_cnt_4w"]].fillna(0)
    item["i_trend"] = item["i_cnt_1w"] / (item["i_cnt_4w"] / 4 + 1)

    out = (cands.join(user, on="cust")
                .join(cust_feat, on="cust")
                .join(item, on="article_id")
                .join(art, on="article_id"))

    # User-item features: does this item fit this customer?
    hg = hist[["cust", "article_id"]].join(art["product_group_code"], on="article_id")
    share = (hg.groupby(["cust", "product_group_code"]).size()
               .div(hg.groupby("cust").size(), level="cust")
               .rename("ui_group_share").reset_index())
    out = out.merge(share, on=["cust", "product_group_code"], how="left")
    out["ui_group_share"] = out["ui_group_share"].fillna(0)
    out["ui_price_ratio"] = out["i_avg_price"] / out["u_avg_price"]
    out["ui_age_gap"] = (out["age"] - out["i_mean_buyer_age"]).abs()

    floats = out.select_dtypes("float64").columns
    out[floats] = out[floats].astype("float32")
    return out

In [28]:
import os, gc, time

# Free anything left over from Phase 4 testing
for name in ["cands0", "hist0", "cands", "train", "test"]:
    globals().pop(name, None)
gc.collect()

for w in [0, 1, 2, 3]:                       # week 0 = test (largest) first, while memory is emptiest
    path = f"/content/feat_week{w}.parquet"
    if os.path.exists(path):
        print(f"week {w}: already saved, skipping")
        continue
    t0 = time.time()
    c, h = build_candidates(recent, label_week=w)
    if w > 0:                                # training weeks: keep customers who bought at least one candidate
        c = c[c.groupby("cust")["label"].transform("max") == 1]
    add_features(c, h, cust_feat, art).to_parquet(path)
    print(f"week {w}: {len(c):,} rows, {c['cust'].nunique():,} customers, "
          f"positive rate {c['label'].mean():.3f} ({time.time() - t0:.0f} s)")
    del c, h; gc.collect()                   # free this week before building the next

if not os.path.exists("/content/test_labels.parquet"):
    recent.loc[recent["week_ago"] == 0, ["cust", "article_id"]].to_parquet("/content/test_labels.parquet")

!ls -lh /content/feat_week*.parquet /content/test_labels.parquet

week 0: 9,064,055 rows, 68,984 customers, positive rate 0.004 (127 s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

week 1: 3,122,873 rows, 23,024 customers, positive rate 0.010 (116 s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

week 2: 3,477,514 rows, 25,704 customers, positive rate 0.010 (117 s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

week 3: 3,622,412 rows, 26,391 customers, positive rate 0.010 (122 s)
-rw-r--r-- 1 root root 112M Nov 24 18:49 /content/feat_week0.parquet
-rw-r--r-- 1 root root  44M Nov 24 18:51 /content/feat_week1.parquet
-rw-r--r-- 1 root root  49M Nov 24 18:53 /content/feat_week2.parquet
-rw-r--r-- 1 root root  51M Nov 24 18:55 /content/feat_week3.parquet
-rw-r--r-- 1 root root 2.3M Nov 24 18:55 /content/test_labels.parquet


Restart Kernel

In [1]:
import time, gc
import numpy as np, pandas as pd, lightgbm as lgb

# Load the 3 training weeks
train = pd.concat([pd.read_parquet(f"/content/feat_week{w}.parquet") for w in [1, 2, 3]], ignore_index=True)
FEATURES = [c for c in train.columns if c not in ("cust", "article_id", "label", "week")]
for c in FEATURES:                                   # safety net: any text column -> number
    if train[c].dtype == "object":
        train[c] = pd.to_numeric(train[c], errors="coerce").astype("float32")

# Negative sampling: keep every bought item + 30% of the rest
rng = np.random.default_rng(42)
keep = (train["label"] == 1) | (rng.random(len(train)) < 0.3)
train = train[keep].sort_values(["week", "cust"], kind="stable").reset_index(drop=True)
groups = train.groupby(["week", "cust"], sort=False).size().to_numpy()
print(f"{len(FEATURES)} features | {len(train):,} rows | {len(groups):,} groups | "
      f"positive rate {train['label'].mean():.3f}")

ranker = lgb.LGBMRanker(
    objective="lambdarank", n_estimators=300, learning_rate=0.05,
    num_leaves=63, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, verbose=-1,
)
t0 = time.time()
ranker.fit(train[FEATURES], train["label"], group=groups)
print(f"trained in {time.time() - t0:.0f} s")

ranker.booster_.save_model("/content/ranker.txt")
del train, ranker; gc.collect()
print("saved /content/ranker.txt")

36 features | 3,140,911 rows | 75,119 groups | positive rate 0.033
trained in 312 s
saved /content/ranker.txt


In [5]:
import json, shutil
from pathlib import Path

KAGGLE_USER = "aparnarameshdatas"          # <- change this to your Kaggle username

ds = Path("/content/kaggle_artifacts")
ds.mkdir(exist_ok=True)
shutil.copy("/content/ranker.txt", ds)

meta = {
    "title": "hm-recommender-artifacts",
    "id": f"{KAGGLE_USER}/hm-recommender-artifacts",
    "licenses": [{"name": "CC0-1.0"}],
}
(ds / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

!kaggle datasets create -p {ds}

Starting upload for file ranker.txt
100% 2.04M/2.04M [00:00<00:00, 3.41MB/s]
Upload successful: ranker.txt (2MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/aparnarameshdatas/hm-recommender-artifacts


In [2]:
!ls -lh /content/ranker.txt /content/results.json

ls: cannot access '/content/results.json': No such file or directory
-rw-r--r-- 1 root root 2.1M Nov 24 19:06 /content/ranker.txt


Restart the Kernel: Evaluate on the Test Week: The Score in Chunks

In [1]:
import sys, os, gc, json, importlib
import numpy as np, pandas as pd, lightgbm as lgb, pyarrow.parquet as pq

# Metrics from your repo (clone first if missing, then import)
if not os.path.exists("/content/repo/src/metrics.py"):
    !rm -rf /content/repo
    !git clone -q https://github.com/WoahImCoding/ecommerce-recommender.git /content/repo
sys.path_importer_cache.pop("/content/repo", None); importlib.invalidate_caches()
if "/content/repo" not in sys.path:
    sys.path.append("/content/repo")
from src.metrics import evaluate

booster = lgb.Booster(model_file="/content/ranker.txt")
FEATURES = booster.feature_name()

parts = []
for batch in pq.ParquetFile("/content/feat_week0.parquet").iter_batches(
        batch_size=1_000_000, columns=["cust", "article_id"] + FEATURES):
    df = batch.to_pandas()
    for c in FEATURES:
        if df[c].dtype == "object":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    parts.append(pd.DataFrame({"cust": df["cust"].to_numpy(),
                               "article_id": df["article_id"].to_numpy(),
                               "score": booster.predict(df[FEATURES].astype("float32")).astype("float32")}))
    del df; gc.collect()
scores = pd.concat(parts, ignore_index=True)
del parts; gc.collect()
print(f"{len(scores):,} candidates scored")

top12 = scores.sort_values(["cust", "score"], ascending=[True, False]).groupby("cust").head(12)
ranked = top12.groupby("cust")["article_id"].apply(list).to_dict()
actual = pd.read_parquet("/content/test_labels.parquet").groupby("cust")["article_id"].apply(set).to_dict()
result = {k: float(v) for k, v in evaluate(lambda c: ranked.get(c, []), actual).items()}
print(len(actual), "test customers")
print("LightGBM ranker:", result)

9,064,055 candidates scored
68984 test customers
LightGBM ranker: {'recall@12': 0.064, 'map@12': 0.0306}


Compare + Feature Importance

In [2]:
summary = pd.DataFrame({
    "Recent popularity (last week)":      {"recall@12": 0.0255, "map@12": 0.0087},
    "Repurchase, then recent popularity": {"recall@12": 0.0525, "map@12": 0.0258},
    "LightGBM ranker":                    result,
}).T
display(summary)

imp = pd.Series(booster.feature_importance("gain"), index=FEATURES)
imp = (100 * imp / imp.sum()).sort_values(ascending=False).round(1)
with open("/content/results.json", "w") as f:
    json.dump({"ranker": result, "test_customers": len(actual),
               "feature_importance_pct": imp.to_dict()}, f, indent=2)
imp.head(15)

,recall@12,map@12
Recent popularity (last week),0.0255,0.0087
"Repurchase, then recent popularity",0.0525,0.0258
LightGBM ranker,0.0640,0.0306


,0
rep_rank,22.4
u_days_since_last,9.3
i_trend,6.3
cop_score,6.1
i_cnt_1w,5.8
ui_price_ratio,3.5
i_mean_buyer_age,3.3
ui_age_gap,3.2
trend_rank,2.9
age,2.9
